# Notebook 2 — Feature Engineering

This notebook turns the raw sales data into the table the demand model learns from:

1. time features and their cyclical versions
2. relative-price features
3. historical demand features (built so they cannot leak the answer)
4. a chronological train / test split and the preprocessing pipeline

In [1]:
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

sys.path.insert(0, "../src")
import pricing_engine as engine   # shared rules used by the notebooks and the app

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

In [2]:
df = pd.read_csv("../data/synthetic_restaurant_sales_data.csv")
df["Date"] = pd.to_datetime(df["Date"])

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (175200, 33)


## 1. Time features

The raw data already carries `Day_of_Week`, `Is_Weekend`, `Season`, `Festival` and `Holiday_Flag`.
Month, quarter, week and day of month are added here. Day of month matters for restaurants: people eat
out more just after payday.

In [3]:
df["Month"] = df["Date"].dt.month
df["Quarter"] = df["Date"].dt.quarter
df["Week"] = df["Date"].dt.isocalendar().week.astype(int)
df["Day_of_Month"] = df["Date"].dt.day

### Cyclical features

December (12) and January (1) are neighbours, but as plain numbers they look far apart. Sine and cosine
put the months and weeks on a circle so the model sees them as neighbours.

In [4]:
df["Month_Sin"] = np.sin(2 * np.pi * df["Month"] / 12)
df["Month_Cos"] = np.cos(2 * np.pi * df["Month"] / 12)

df["Week_Sin"] = np.sin(2 * np.pi * df["Week"] / 52)
df["Week_Cos"] = np.cos(2 * np.pi * df["Week"] / 52)

## 2. Price features

A Rs 20 change means something very different on a Rs 35 roti and a Rs 445 biryani, so price is
expressed relative to the item's own base price and to the competitor's price.

`Selling_Price` is the price the customer actually pays, after any discount.

In [5]:
df["Price_Index"] = df["Selling_Price"] / df["Base_Price"]

df["Competitor_Price_Gap"] = df["Selling_Price"] - df["Competitor_Price"]

df["Competitor_Price_Ratio"] = df["Selling_Price"] / df["Competitor_Price"]

df[["Base_Price", "Selling_Price", "Competitor_Price", "Price_Index", "Competitor_Price_Gap", "Competitor_Price_Ratio"]].describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
Base_Price,175200.0,209.875,113.295,30.000,118.750,197.500,301.250,500.000
Selling_Price,175200.0,203.157,111.434,22.000,113.000,184.000,290.000,550.000
Competitor_Price,175200.0,207.084,112.959,21.000,114.000,188.000,299.000,521.000
Price_Index,175200.0,0.968,0.084,0.708,0.900,0.991,1.045,1.109
Competitor_Price_Gap,175200.0,-3.927,26.322,-145.000,-16.000,-3.000,8.000,164.000
Competitor_Price_Ratio,175200.0,0.988,0.118,0.647,0.913,0.982,1.056,1.633


## 3. Historical demand features (leakage-safe)

How well an item normally sells at an outlet is the best single predictor of tomorrow's sales.

To avoid leakage the average for a day uses **only earlier days of the same item at the same outlet**:
the data is sorted by date inside each item-outlet pair and `shift(1)` pushes the values down one day
before the running average is taken.

In [6]:
df = df.sort_values(["Item_ID", "Outlet_ID", "Date"]).reset_index(drop=True)

grouped = df.groupby(["Item_ID", "Outlet_ID"])

df["Avg_Item_Demand"] = grouped["Units_Sold"].transform(lambda x: x.shift(1).expanding().mean())
df["Avg_Item_Price"] = grouped["Selling_Price"].transform(lambda x: x.shift(1).expanding().mean())

df = df.sort_values(["Date", "Outlet_ID", "Item_ID"]).reset_index(drop=True)

print("Leakage-safe historical features created.")

Leakage-safe historical features created.


In [7]:
example = df[(df["Item_ID"] == "M06") & (df["Outlet_ID"] == "R01")]
example[["Date", "Item_Name", "Outlet_ID", "Units_Sold", "Avg_Item_Demand", "Selling_Price", "Avg_Item_Price"]].head(6)

,Date,Item_Name,Outlet_ID,Units_Sold,Avg_Item_Demand,Selling_Price,Avg_Item_Price
5,2024-10-01,Butter Chicken,R01,56,NaN,353,NaN
245,2024-10-02,Butter Chicken,R01,64,56.000000,353,353.0
485,2024-10-03,Butter Chicken,R01,62,60.000000,353,353.0
725,2024-10-04,Butter Chicken,R01,82,60.666667,353,353.0
965,2024-10-05,Butter Chicken,R01,92,66.000000,353,353.0
1205,2024-10-06,Butter Chicken,R01,73,71.200000,415,353.0


The first day of every item-outlet pair has no history, so its average is empty. Those rows are removed.

In [8]:
before_rows = len(df)
df = df.dropna(subset=["Avg_Item_Demand", "Avg_Item_Price"]).reset_index(drop=True)

print("Rows removed:", before_rows - len(df))
print("Final shape :", df.shape)

Rows removed: 240
Final shape : (174960, 46)


## 4. Consistency check with the app

The Streamlit app builds its model inputs with `src/pricing_engine.py`. The check below rebuilds the
features with that file and confirms they are identical to the ones created above, so the app can never
feed the model something different from what it was trained on.

In [9]:
raw = pd.read_csv("../data/synthetic_restaurant_sales_data.csv")
raw["Date"] = pd.to_datetime(raw["Date"])

engine_df = engine.add_history_features(engine.add_price_features(engine.add_time_features(raw)))
engine_df = engine_df.dropna(subset=["Avg_Item_Demand", "Avg_Item_Price"]).reset_index(drop=True)

engineered = ["Month", "Quarter", "Week", "Day_of_Month", "Month_Sin", "Month_Cos", "Week_Sin", "Week_Cos",
              "Price_Index", "Competitor_Price_Gap", "Competitor_Price_Ratio", "Avg_Item_Demand", "Avg_Item_Price"]

print("Same rows   :", len(engine_df) == len(df))
print("Same values :", bool(np.allclose(engine_df[engineered].to_numpy(dtype=float), df[engineered].to_numpy(dtype=float))))

Same rows   : True
Same values : True


## 5. Target and features

The target is `Units_Sold`. Revenue, profit and the cost columns are outcomes of the sale (or are
only needed for the profit calculation), so they are kept out of the model.

In [10]:
target = "Units_Sold"

non_feature_columns = engine.NON_FEATURE_COLUMNS
categorical_features = engine.CATEGORICAL_FEATURES
numerical_features = engine.NUMERICAL_FEATURES

X = df[categorical_features + numerical_features]
y = df[target]

print("Columns kept out of the model:", non_feature_columns)
print("\nCategorical features:", len(categorical_features))
print(categorical_features)
print("\nNumerical features:", len(numerical_features))
print(numerical_features)
print("\nX shape:", X.shape, "| y shape:", y.shape)

Columns kept out of the model: ['Date', 'Item_Name', 'Units_Sold', 'Revenue', 'Profit', 'Unit_Cost', 'Overhead_Cost', 'Promotion_Cost', 'Total_Cost']

Categorical features: 11
['Day_of_Week', 'Season', 'Festival', 'Item_ID', 'Category', 'Food_Type', 'Outlet_ID', 'City', 'Outlet_Type', 'Location_Type', 'Promotion_Type']

Numerical features: 26
['Is_Weekend', 'Holiday_Flag', 'Footfall_Index', 'Base_Price', 'Selling_Price', 'Discount_Percentage', 'Promotion_Flag', 'Promotion_Depth', 'Competitor_Price', 'Competitor_Promotion_Flag', 'Competitor_Discount', 'Temperature', 'Rainfall', 'Month', 'Quarter', 'Week', 'Day_of_Month', 'Month_Sin', 'Month_Cos', 'Week_Sin', 'Week_Cos', 'Price_Index', 'Competitor_Price_Gap', 'Competitor_Price_Ratio', 'Avg_Item_Demand', 'Avg_Item_Price']

X shape: (174960, 37) | y shape: (174960,)


## 6. Chronological train / test split

Sales data is a time series, so the model is trained on the first 80% of dates and tested on the
last 20%. A random split would let the model peek at the future.

In [11]:
unique_dates = np.sort(df["Date"].unique())
split_date = unique_dates[int(len(unique_dates) * 0.80)]

train_mask = df["Date"] < split_date
test_mask = df["Date"] >= split_date

X_train, X_test = X.loc[train_mask], X.loc[test_mask]
y_train, y_test = y.loc[train_mask], y.loc[test_mask]

print("Split date:", pd.Timestamp(split_date).date())
print("Training observations:", len(X_train))
print("Testing observations :", len(X_test))
print("Training period:", df.loc[train_mask, "Date"].min().date(), "to", df.loc[train_mask, "Date"].max().date())
print("Testing period :", df.loc[test_mask, "Date"].min().date(), "to", df.loc[test_mask, "Date"].max().date())

Split date: 2026-05-08
Training observations: 139920
Testing observations : 35040
Training period: 2024-10-02 to 2026-05-07
Testing period : 2026-05-08 to 2026-09-30


## 7. Preprocessing pipeline

Text columns are one-hot encoded; numeric columns pass through unchanged (tree models do not need scaling).

In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
        ("numerical", "passthrough", numerical_features),
    ],
    verbose_feature_names_out=False,
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Original training shape :", X_train.shape)
print("Processed training shape:", X_train_processed.shape)
print("Original testing shape  :", X_test.shape)
print("Processed testing shape :", X_test_processed.shape)

Original training shape : (139920, 37)
Processed training shape: (139920, 115)
Original testing shape  : (35040, 37)
Processed testing shape : (35040, 115)


## 8. Save the processed dataset

In [13]:
df.to_csv("../data/processed_restaurant_sales_data.csv.gz", index=False, compression="gzip")

print("Processed dataset saved successfully!")
print("Shape:", df.shape)

Processed dataset saved successfully!
Shape: (174960, 46)
